# Mastering Hybrid Retrieval: Combining Keyword and Semantic Search for Advanced RAG Systems

Retrieval-Augmented Generation (RAG) systems are foundational to modern LLM applications, allowing models to ground their answers in specific, verifiable knowledge bases. However, relying solely on one type of retrieval mechanism—either keyword matching or semantic embedding—creates significant blind spots. Keyword search (like BM25) is excellent at finding exact terminology but fails when the user uses synonyms or discusses related concepts. Conversely, dense vector search excels at understanding meaning and context but can struggle with highly specific jargon or proper nouns that lack sufficient surrounding context for embeddings to capture.

This notebook addresses this critical limitation by implementing **Hybrid Retrieval**. We demonstrate how to combine the strengths of sparse keyword models (which guarantee recall on exact terms) with the contextual power of dense vector search (which guarantees semantic relevance). By using an `EnsembleRetriever` and techniques like Reciprocal Rank Fusion (RRF), we build a robust, multi-faceted retrieval layer. This is crucial for advanced RAG pipelines because real-world queries are rarely simple; they often require both specific terminology *and* deep contextual understanding to be fully answered.

By the end of this exercise, you will not only understand how these different retrievers function but also how to architect a production-grade retrieval component that maximizes recall and precision simultaneously—a non-negotiable requirement for building reliable, enterprise-level LLM applications using frameworks like LangGraph.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Differentiate Retrieval Methods:** Articulate the core differences between sparse keyword search (BM25) and dense vector search (OpenAI Embeddings/Chroma).
*   **Identify Limitations:** Analyze a knowledge base and predict where single-method retrievers will fail (e.g., missing semantic matches or failing on exact jargon).
*   **Implement Hybrid Retrieval:** Construct an `EnsembleRetriever` to combine multiple retrieval sources, leveraging techniques like Reciprocal Rank Fusion (RRF) for superior results.
*   **Optimize RAG Pipelines:** Understand how integrating hybrid search improves the overall robustness and accuracy of a LangGraph-powered RAG agent.


### Setup and Initialization

This cell sets up the necessary environment by loading API keys from a `.env` file. It imports key components for advanced retrieval, including OpenAI embeddings, Chroma vector store, BM25 (a keyword-based retriever), and `EnsembleRetriever` to combine different search methods.


In [4]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Load OPENAI_API_KEY from .env file to ensure credentials are available for API calls
load_dotenv()


True

### Document Setup and Data Preparation

This cell initializes the core dataset (`docs`) used throughout the notebook. It creates a list of `Document` objects, each containing page content and metadata (like topic). This structured data is crucial for testing advanced retrieval techniques, as it contains documents specifically designed to test keyword matching vs. semantic understanding.


In [5]:
# 12 documents spanning health, programming, history, and nature
# Docs 1-2: contain the exact word "vaccine" — BM25 keyword match
# Docs 3-5: semantically related (immune system, antibodies, herd immunity) but lack the word "vaccine"
#            — dense search finds these, BM25 misses them
# Docs 6-12: off-topic
docs = [
    Document(page_content="Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.", metadata={"topic": "health"}),
    Document(page_content="The flu vaccine is reformulated each year to match the most prevalent circulating virus strains.", metadata={"topic": "health"}),
    Document(page_content="The immune system produces antibodies that recognise and neutralise foreign pathogens in the body.", metadata={"topic": "health"}),
    Document(page_content="Herd immunity occurs when enough of a population becomes resistant to a disease, slowing its spread.", metadata={"topic": "health"}),
    Document(page_content="White blood cells called B-lymphocytes produce proteins that bind to and destroy specific antigens.", metadata={"topic": "health"}),
    Document(page_content="Version control systems like Git track changes to code and enable collaboration across teams.", metadata={"topic": "programming"}),
    Document(page_content="Docker containers package applications with their dependencies for consistent deployment.", metadata={"topic": "programming"}),
    Document(page_content="The French Revolution began in 1789 and fundamentally transformed European political structures.", metadata={"topic": "history"}),
    Document(page_content="The Silk Road was an ancient trade network connecting China to the Mediterranean world.", metadata={"topic": "history"}),
    Document(page_content="The Amazon rainforest produces about 20% of the world's oxygen and houses 10% of all species.", metadata={"topic": "nature"}),
    Document(page_content="Coral reefs cover less than 1% of the ocean floor but support about 25% of all marine species.", metadata={"topic": "nature"}),
    Document(page_content="REST APIs communicate over HTTP using standard methods like GET, POST, PUT, and DELETE.", metadata={"topic": "programming"}),
]

### Hybrid Retriever Setup

This cell initializes two distinct retrieval mechanisms: a dense retriever (using ChromaDB and OpenAI embeddings) for semantic similarity, and a sparse retriever (using BM25Plus) for keyword matching. By setting up both, we prepare the system to perform hybrid search, combining the strengths of vector embeddings and traditional indexing.


In [6]:
# Dense retriever: ChromaDB with OpenAI embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create a persistent vector store (ChromaDB) from the documents using the specified embeddings.
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="hybrid_search",
)

# Convert the vector store into a retriever object, configured to fetch the top 4 most similar chunks.
chroma_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

# Sparse retriever: BM25Plus variant on raw text, no embeddings
# BM25Plus ensures every matched term contributes a positive score,
# which improves recall for short documents like the ones we have here
# Initialize the sparse retriever using BM25 (a keyword-based method).
bm25_retriever = BM25Retriever.from_documents(
    docs,
    k=2,
    bm25_variant="plus"
)


### Ensemble Retrieval using RRF

The `EnsembleRetriever` class combines the outputs of multiple specialized retrievers (like vector and keyword search) into a single, robust component. It uses Reciprocal Rank Fusion (RRF) to intelligently merge results, giving more weight to the Chroma retriever while still incorporating BM25's unique strengths.


In [11]:
# EnsembleRetriever merges results from both retrievers using Reciprocal Rank Fusion (RRF)

ensemble_retriever = EnsembleRetriever(
    # List of individual retrievers to combine (e.g., vector and keyword search)
    retrievers=[chroma_retriever, bm25_retriever],
    # Weights determine the influence of each retriever's results on the final ranking
    weights=[0.8, 0.2]
)



### Hybrid Retrieval Demonstration

This cell demonstrates the core concept of hybrid search by querying three different retrieval mechanisms (BM25, ChromaDB, and an Ensemble retriever) with the same query. It visually compares how keyword-based matching (BM25), purely semantic matching (ChromaDB), and combined approaches (Ensemble) retrieve documents, highlighting the benefits of combining methods for comprehensive context gathering.


In [12]:
query = "How do vaccines work to protect against diseases?"

# Invoke the specialized retrievers with the query.
bm25_results = bm25_retriever.invoke(query)
chroma_results = chroma_retriever.invoke(query)
ensemble_results = ensemble_retriever.invoke(query)

# --- BM25 Demonstration ---
# BM25 matches on the exact word "vaccine" — finds docs 1 and 2
# but misses the semantically related immune/antibody docs (3, 4, 5)
print("=== BM25 Only (keyword match) ===")
for i, doc in enumerate(bm25_results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")

print()

# --- ChromaDB Demonstration ---
# Dense search finds docs 3, 4, 5 through semantic understanding
# even though they don't contain the word "vaccine"
print("=== ChromaDB Only (semantic match) ===")
for i, doc in enumerate(chroma_results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")

print()

# --- Ensemble Demonstration ---
# Ensemble combines both — surfaces keyword matches AND semantic matches
print("=== Ensemble / Hybrid (keyword + semantic) ===")
for i, doc in enumerate(ensemble_results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")


=== BM25 Only (keyword match) ===
  [1] topic=health: Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.
  [2] topic=programming: REST APIs communicate over HTTP using standard methods like GET, POST, PUT, and DELETE.

=== ChromaDB Only (semantic match) ===
  [1] topic=health: Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.
  [2] topic=health: The immune system produces antibodies that recognise and neutralise foreign pathogens in the body.
  [3] topic=health: The flu vaccine is reformulated each year to match the most prevalent circulating virus strains.
  [4] topic=health: White blood cells called B-lymphocytes produce proteins that bind to and destroy specific antigens.

=== Ensemble / Hybrid (keyword + semantic) ===
  [1] topic=health: Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.
  [2] topic=health: The immune system produces antibodies that r